In [1]:
# ================================
# 09-hybrid-comprehensive-final.ipynb
# Final hybrid evaluation – runs only the missing configurations
# Budgets: 50 (new), 100 (Telugu missing), 20000 (both missing)
# LRs: 0.0005 (budget 50), 0.001 (budget 100), 0.0003 (budget 20000)
# Seeds: 42, 123, 456
# ================================

# ------------------------------
# Environment setup
# ------------------------------
!pip install -q bitsandbytes
!pip uninstall -y torchao
!pip install -q peft --no-deps
!pip install -q trl --no-deps
!pip install -q accelerate

import torch, transformers, datasets, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# Quick CUDA sanity check
x = torch.randn(100).cuda()
y = torch.randn(100).cuda()
z = torch.matmul(x, y)
print("✓ CUDA ops work:", z.item())

# ------------------------------
# Imports
# ------------------------------
import os, time, math, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, IA3Config, TaskType, get_peft_model
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
from torch.optim.lr_scheduler import LinearLR

# ------------------------------
# Configuration: missing runs only
# ------------------------------
MODEL_NAME = "xlm-roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 128
BATCH_SIZE = 16
SEEDS = [42, 123, 456]

# Missing configurations:
# Format: (language, budget, lr, epochs)
# - Budget 50: new, LR=0.0005 (conservative)
# - Budget 100: Telugu missing, LR=0.001 (from Hindi best)
# - Budget 20000: both languages missing, LR=0.0003 (best from 500-2000)
MISSING_RUNS = [
    ("hi", 50,    0.0005, 10),
    ("te", 50,    0.0005, 10),
    ("te", 100,   0.001,  10),
    ("hi", 20000, 0.0003, 3),
    ("te", 20000, 0.0003, 3),
]

# Paths
DATA_ROOT = "/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ------------------------------
# Helper functions (same as before)
# ------------------------------
def load_base_model():
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).cuda()

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def build_loaders(language, budget):
    train_df = pd.read_parquet(f"{DATA_ROOT}/{language}/train_{budget}.parquet")
    valid_df = pd.read_parquet(f"{DATA_ROOT}/{language}/valid.parquet")
    def tokenize(batch):
        return tokenizer(batch["premise"], batch["hypothesis"],
                         truncation=True, padding="max_length", max_length=MAX_LENGTH)
    train_ds = Dataset.from_pandas(train_df).rename_column("label", "labels")
    valid_ds = Dataset.from_pandas(valid_df).rename_column("label", "labels")
    train_ds = train_ds.map(tokenize, batched=True)
    valid_ds = valid_ds.map(tokenize, batched=True)
    keep = ["input_ids", "attention_mask", "labels"]
    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
    valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in keep])
    train_ds.set_format("torch")
    valid_ds.set_format("torch")
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=32)
    return train_loader, valid_loader

def build_hybrid_model():
    base = load_base_model()
    ia3_config = IA3Config(
        task_type=TaskType.SEQ_CLS,
        target_modules=["key", "value", "output.dense"],
        feedforward_modules=["output.dense"],
        modules_to_save=["classifier"]
    )
    model = get_peft_model(base, ia3_config)
    lora_config = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.1, bias="none",
        task_type=TaskType.SEQ_CLS,
        target_modules=["query", "value"],
        modules_to_save=["classifier"]
    )
    model = get_peft_model(model, lora_config)
    return model

def train_and_evaluate(method, language, budget, seed, lr, epochs):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    train_loader, valid_loader = build_loaders(language, budget)
    model = build_hybrid_model()
    trainable_params = count_trainable_params(model)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    steps_per_epoch = math.ceil(budget / BATCH_SIZE)
    total_steps = steps_per_epoch * epochs
    warmup_steps = int(0.1 * total_steps)
    scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_steps)
    model.train()
    start_time = time.perf_counter()
    for epoch in range(epochs):
        for batch in train_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["labels"])
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
    train_time = time.perf_counter() - start_time
    torch.cuda.reset_peak_memory_stats()
    model.train()
    with torch.no_grad():
        dummy_batch = next(iter(train_loader))
        dummy_batch = {k: v.cuda() for k, v in dummy_batch.items()}
        _ = model(**dummy_batch)
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
    model.eval()
    predictions, labels = [], []
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())
    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")
    del model
    torch.cuda.empty_cache()
    return {
        "method": method,
        "language": language,
        "budget": budget,
        "seed": seed,
        "lr": lr,
        "epochs": epochs,
        "warmup_steps": warmup_steps,
        "total_steps": total_steps,
        "accuracy": round(accuracy, 6),
        "macro_f1": round(macro_f1, 6),
        "trainable_params": trainable_params,
        "peak_gpu_memory_gb": round(peak_memory_gb, 4),
        "training_time_sec": round(train_time, 2)
    }

# ------------------------------
# Run only missing configurations
# ------------------------------
results_file = "/kaggle/working/hybrid_missing_runs.csv"
total_runs = sum(len(SEEDS) for _ in MISSING_RUNS)
current_run = 0

for lang, budget, lr, epochs in MISSING_RUNS:
    for seed in SEEDS:
        current_run += 1
        print(f"\n[{current_run}/{total_runs}] Hybrid | {lang} | budget={budget} | lr={lr} | seed={seed}")
        try:
            result = train_and_evaluate("ia3_lora_sequential", lang, budget, seed, lr, epochs)
            print(f"  ✓ Acc: {result['accuracy']:.4f}  F1: {result['macro_f1']:.4f}  Time: {result['training_time_sec']:.1f}s")
            pd.DataFrame([result]).to_csv(results_file, mode='a', header=not os.path.exists(results_file), index=False)
        except Exception as e:
            print(f"  ✗ ERROR: {e}")
            pd.DataFrame([{"method":"ia3_lora_sequential", "language":lang, "budget":budget, "seed":seed, "lr":lr, "error":str(e)}]).to_csv(results_file, mode='a', header=not os.path.exists(results_file), index=False)

print(f"\n✅ Missing runs completed. {total_runs} configurations attempted.")

# ------------------------------
# Load all results and produce master table
# ------------------------------
# Load pure results (from 108-run sweep)
pure_file = "/kaggle/input/05-full-experiment-sweep/experiment_results.csv"
if not os.path.exists(pure_file):
    pure_file = "/kaggle/working/experiment_results.csv"
pure_df = pd.read_csv(pure_file) if os.path.exists(pure_file) else pd.DataFrame()

# Load previous hybrid results (Hindi and Telugu we already have)
prev_hybrid_files = [
    "/kaggle/working/all_hybrid_results.csv",
    "/kaggle/input/hybrid/all_hybrid_results.csv",
    "/kaggle/working/hybrid_te_results.csv",
    "/kaggle/input/hybrid/hybrid_te_results.csv"
]
prev_hybrid = pd.DataFrame()
for f in prev_hybrid_files:
    if os.path.exists(f):
        prev_hybrid = pd.concat([prev_hybrid, pd.read_csv(f)], ignore_index=True)

# Load new missing runs
if os.path.exists(results_file):
    new_hybrid = pd.read_csv(results_file)
else:
    new_hybrid = pd.DataFrame()

# Combine all hybrid
hybrid_all = pd.concat([prev_hybrid, new_hybrid], ignore_index=True)
hybrid_all = hybrid_all[hybrid_all["language"].isin(["hi", "te"])]
hybrid_all.to_csv("/kaggle/working/all_hybrid_final.csv", index=False)

# Master aggregation
if not pure_df.empty and not hybrid_all.empty:
    pure_sub = pure_df[pure_df["method"].isin(["lora", "ia3"])]
    pure_agg = pure_sub.groupby(["method", "language", "budget"]).agg(
        mean_f1=("macro_f1", "mean"),
        max_f1=("macro_f1", "max"),
        collapses=("accuracy", lambda x: sum(np.isclose(x, 1/3, atol=0.001)))
    ).reset_index()
    hybrid_agg = hybrid_all.groupby(["language", "budget"]).agg(
        mean_f1=("macro_f1", "mean"),
        max_f1=("macro_f1", "max"),
        collapses=("accuracy", lambda x: sum(np.isclose(x, 1/3, atol=0.001)))
    ).reset_index()
    hybrid_agg["method"] = "hybrid"
    master = pd.concat([pure_agg, hybrid_agg], ignore_index=True)

    # Pivot tables
    pivot_mean = master.pivot_table(index=["language", "budget"], columns="method", values="mean_f1").round(4)
    pivot_max = master.pivot_table(index=["language", "budget"], columns="method", values="max_f1").round(4)
    pivot_collapses = master.pivot_table(index=["language", "budget"], columns="method", values="collapses").fillna(0).astype(int)

    print("\n=== MASTER MEAN F1 ===")
    print(pivot_mean)
    print("\n=== MASTER BEST F1 ===")
    print(pivot_max)
    print("\n=== MASTER COLLAPSES ===")
    print(pivot_collapses)

    # Save to CSV
    pivot_mean.to_csv("/kaggle/working/master_mean_f1.csv")
    pivot_max.to_csv("/kaggle/working/master_max_f1.csv")
    pivot_collapses.to_csv("/kaggle/working/master_collapses.csv")

    # ------------------------------
    # Plot learning curves
    # ------------------------------
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    colors = {"lora": "#1f77b4", "ia3": "#ff7f0e", "hybrid": "#2ca02c"}

    for ax, lang in zip(axes, ["hi", "te"]):
        for method in ["lora", "ia3", "hybrid"]:
            sub = master[(master["language"]==lang) & (master["method"]==method)]
            sub = sub.sort_values("budget")
            if not sub.empty:
                ax.plot(sub["budget"], sub["mean_f1"], marker="o", label=method.upper(), color=colors.get(method, "gray"))
        ax.set_xscale("log")
        ax.set_xlabel("Training budget (samples)")
        ax.set_title(f"{lang.upper()}")
        ax.axhline(0.333, color="gray", linestyle="--", linewidth=1, label="Random chance")
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("Macro F1")
    axes[0].legend(loc="lower right")
    plt.tight_layout()
    plt.savefig("/kaggle/working/hybrid_vs_pure_learning_curves.png", dpi=150)
    plt.show()

else:
    print("Pure results or hybrid results missing – check file paths.")

print("\nAll done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 42.3 MB/s eta 0:00:00:00:0100:01
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 13.5 MB/s eta 0:00:00 0:00:01
torch: 2.10.0+cu128
transformers: 5.0.0
peft: 0.19.1
CUDA available: True
GPU: Tesla T4


✓ CUDA ops work: -5.776362419128418


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a mode

  ✓ Acc: 0.6659  F1: 0.6668  Time: 875.4s

✅ Missing runs completed. 15 configurations attempted.
Pure results or hybrid results missing – check file paths.

All done.
